# Module 7 — NumPy: Numerical Computing

**Python for Machine Learning · MSc Artificial Intelligence · Swansea University**

---

## Learning objectives

1. Explain why NumPy arrays are much faster than Python lists.
2. Create arrays and understand `shape`, `ndim`, `dtype`, `reshape`.
3. Index and slice multi-dimensional arrays, use boolean masks and fancy indexing; know **views vs copies**.
4. Write **vectorised** code and apply the rules of **broadcasting**.
5. Aggregate along **axes**, use linear algebra and random number generation.
6. Implement ML building blocks (standardisation, distances, softmax, linear regression) with NumPy.

---

## 7.1 Why NumPy?

NumPy provides the `ndarray`: an **n-dimensional array of elements of one type**, stored in a contiguous block of memory. Operations on arrays run in optimised C code instead of the Python interpreter. Every ML library — pandas, scikit-learn, Matplotlib, and (conceptually) PyTorch tensors — is built on this idea.

In [1]:
import numpy as np
import time

n = 1_000_000
py_list = list(range(n))
arr = np.arange(n)

start = time.perf_counter()
squared_list = [x ** 2 for x in py_list]
t_list = time.perf_counter() - start

start = time.perf_counter()
squared_arr = arr ** 2
t_arr = time.perf_counter() - start

print(f"list: {t_list:.4f} s, numpy: {t_arr:.4f} s -> ~{t_list / t_arr:.0f}x faster")

list: 0.0536 s, numpy: 0.0017 s -> ~31x faster


The style of writing whole-array operations instead of explicit Python loops is called **vectorisation**. It is the single most important habit for efficient ML code.

## 7.2 Creating arrays

In [2]:
a = np.array([1, 2, 3])                     # from a list
M = np.array([[1, 2, 3], [4, 5, 6]])        # 2-D from a list of lists
print(a)
print(M)
print("shape:", M.shape, "ndim:", M.ndim, "size:", M.size, "dtype:", M.dtype)

[1 2 3]
[[1 2 3]
 [4 5 6]]
shape: (2, 3) ndim: 2 size: 6 dtype: int64


In [3]:
print(np.zeros((2, 3)))           # all zeros, shape (2, 3)
print(np.ones(4))
print(np.full((2, 2), 7.0))
print(np.eye(3))                  # identity matrix
print(np.arange(0, 10, 2))        # like range(), but returns an array
print(np.linspace(0, 1, 5))       # 5 evenly spaced points from 0 to 1 INCLUDING 1
print(np.zeros_like(M))           # same shape/dtype as M

[[0. 0. 0.]
 [0. 0. 0.]]
[1. 1. 1. 1.]
[[7. 7.]
 [7. 7.]]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]
[[0 0 0]
 [0 0 0]]


### Terminology for shapes

| Name | ndim | Example shape | ML example |
|---|---|---|---|
| scalar | 0 | `()` | a loss value |
| vector | 1 | `(n,)` | labels `y`, weights `w` |
| matrix | 2 | `(n_samples, n_features)` | design matrix `X` |
| tensor | 3+ | `(n_images, height, width)` | a batch of grayscale images |

> **Convention:** in scikit-learn, `X` always has shape `(n_samples, n_features)` — one **row** per sample, one **column** per feature.

### Data types

All elements share one `dtype`. Common ones: `int64`, `float64` (the default for real numbers), `float32` (common in deep learning, half the memory), `bool`.

In [4]:
x = np.array([1, 2, 3])
print(x.dtype)
y = x.astype(np.float32)
print(y, y.dtype, y.nbytes, "bytes")

mixed = np.array([1, 2.5, 3])     # upcast: everything becomes float
print(mixed.dtype)

ints = np.array([1, 2, 3])
ints[0] = 9.99                    # silently truncated, because the array's dtype is int!
print(ints)

int64
[1. 2. 3.] float32 12 bytes
float64
[9 2 3]


## 7.3 Reshaping

In [5]:
a = np.arange(12)
print(a.reshape(3, 4))
print(a.reshape(2, -1))           # -1 means "work out this dimension for me"
print(a.reshape(3, 4).T)          # transpose: shape (4, 3)
print(a.reshape(3, 4).ravel())    # flatten back to 1-D

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
[[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]]
[[ 0  4  8]
 [ 1  5  9]
 [ 2  6 10]
 [ 3  7 11]]
[ 0  1  2  3  4  5  6  7  8  9 10 11]


1-D arrays have shape `(n,)`, which is **not** the same as a column `(n, 1)` or a row `(1, n)`. scikit-learn requires 2-D `X`, so a single feature must be reshaped:

In [6]:
feature = np.array([1.0, 2.0, 3.0])
print(feature.shape)
print(feature.reshape(-1, 1).shape)     # column vector, shape (3, 1)
print(feature[:, np.newaxis].shape)     # same thing using np.newaxis
print(feature[np.newaxis, :].shape)     # row vector, shape (1, 3)

(3,)
(3, 1)
(3, 1)
(1, 3)


## 7.4 Indexing and slicing

### 1-D: same as lists

In [7]:
a = np.arange(10, 20)
print(a[0], a[-1], a[2:5], a[::3])

10 19 [12 13 14] [10 13 16 19]


### 2-D: `array[row, column]`

In [8]:
M = np.arange(1, 13).reshape(3, 4)
print(M)
print(M[1, 2])        # row 1, column 2
print(M[0])           # first row
print(M[:, 1])        # second COLUMN (all rows)
print(M[:2, 1:3])     # rows 0-1, columns 1-2
print(M[-1, ::-1])    # last row, reversed

[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
7
[1 2 3 4]
[ 2  6 10]
[[2 3]
 [6 7]]
[12 11 10  9]


### Boolean masking — filtering data

A comparison produces a **boolean array**; using it as an index keeps only the `True` positions.

In [9]:
scores = np.array([55, 78, 91, 42, 67, 88])
mask = scores >= 60
print(mask)
print(scores[mask])
print(scores[(scores >= 60) & (scores < 90)])   # combine with & (and), | (or), ~ (not)
print(mask.sum(), mask.mean())                  # True counts as 1: count and fraction

[False  True  True False  True  True]
[78 91 67 88]
[78 67 88]
4 0.6666666666666666


> Use `&`, `|`, `~` with arrays (not `and`, `or`, `not`), and **always use parentheses** around each comparison.

Masks can also *assign*:

In [10]:
x = np.array([-2.0, 3.0, -1.0, 5.0])
x[x < 0] = 0          # ReLU, in place
print(x)
print(np.where(scores >= 60, "pass", "fail"))   # vectorised if/else

[0. 3. 0. 5.]
['fail' 'pass' 'pass' 'fail' 'pass' 'pass']


### Fancy indexing — selecting with integer arrays

In [11]:
a = np.array([10, 20, 30, 40, 50])
print(a[[0, 3, 4]])

rng = np.random.default_rng(0)
idx = rng.permutation(len(a))      # a random order of indices: used for shuffling data
print(idx, a[idx])

[10 40 50]
[2 4 3 0 1] [30 50 40 10 20]


Shuffling `X` and `y` with the **same** permutation keeps samples and labels aligned:

In [12]:
X = np.arange(10).reshape(5, 2)
y = np.array([0, 1, 0, 1, 1])
perm = rng.permutation(len(y))
X_shuf, y_shuf = X[perm], y[perm]
print(X_shuf, y_shuf)

[[8 9]
 [2 3]
 [4 5]
 [0 1]
 [6 7]] [1 1 0 0 1]


### ⚠️ Views versus copies

**Slicing returns a view**: a new array object that *shares the same memory*. Modifying the view modifies the original. **Boolean and fancy indexing return copies.**

In [13]:
a = np.arange(6)
view = a[1:4]
view[0] = 100
print("a after modifying the view:", a)

b = np.arange(6)
safe = b[1:4].copy()   # explicit copy
safe[0] = 100
print("b is unchanged:", b)

c = np.arange(6)
fancy = c[[1, 2, 3]]   # fancy indexing -> copy
fancy[0] = 100
print("c is unchanged:", c)
print(np.shares_memory(a, view), np.shares_memory(c, fancy))

a after modifying the view: [  0 100   2   3   4   5]
b is unchanged: [0 1 2 3 4 5]
c is unchanged: [0 1 2 3 4 5]
True False


Views make NumPy fast and memory-efficient (no data is duplicated), but can cause subtle bugs. When in doubt, use `.copy()`.

## 7.5 Vectorised operations and universal functions

Arithmetic operators work **element-wise**:

In [14]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([10.0, 20.0, 30.0])
print(a + b, a * b, b / a, a ** 2)
print(a * 10 + 1)

[11. 22. 33.] [10. 40. 90.] [10. 10. 10.] [1. 4. 9.]
[11. 21. 31.]


**Universal functions** (ufuncs) apply a mathematical function element-wise:

In [15]:
x = np.linspace(-2, 2, 5)
print(np.exp(x))
print(np.log(np.array([1, np.e, np.e ** 2])))
print(np.sqrt(np.array([4, 9, 16])))
print(np.abs(x), np.round(np.sin(x), 3))
print(np.maximum(x, 0))     # element-wise max: ReLU again
print(np.clip(x, -1, 1))    # limit values to [-1, 1]

[0.13533528 0.36787944 1.         2.71828183 7.3890561 ]
[0. 1. 2.]
[2. 3. 4.]
[2. 1. 0. 1. 2.] [-0.909 -0.841  0.     0.841  0.909]
[0. 0. 0. 1. 2.]
[-1. -1.  0.  1.  1.]


Example: the **sigmoid** function $\sigma(z) = 1/(1+e^{-z})$ for a whole array at once:

In [16]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

print(sigmoid(np.array([-5, -1, 0, 1, 5])))

[0.00669285 0.26894142 0.5        0.73105858 0.99330715]


## 7.6 Broadcasting

Broadcasting lets NumPy combine arrays of **different shapes** without copying data. Shapes are compared from the **right**; two dimensions are compatible if they are **equal** or one of them is **1**. A dimension of size 1 is (virtually) stretched to match.

```text
X        (4, 3)        X        (4, 3)        a   (3, 1)
mu          (3,)       col   (4, 1)           b      (4,)
------------------     ---------------        ---------------
result   (4, 3) ✓      result (4, 3) ✓        result (3, 4) ✓  (outer)

X (4, 3) + v (4,)  ->  compare 3 vs 4  ->  ERROR
```

In [17]:
X = np.array([[1.0, 200.0, 3.0],
              [2.0, 180.0, 5.0],
              [3.0, 220.0, 4.0],
              [4.0, 210.0, 8.0]])

mu = X.mean(axis=0)           # mean of each column (feature), shape (3,)
sigma = X.std(axis=0)
X_std = (X - mu) / sigma      # (4,3) - (3,) -> broadcast across rows
print(mu)
print(X_std.round(3))
print(X_std.mean(axis=0).round(10), X_std.std(axis=0))

[  2.5 202.5   5. ]
[[-1.342 -0.169 -1.069]
 [-0.447 -1.521  0.   ]
 [ 0.447  1.183 -0.535]
 [ 1.342  0.507  1.604]]
[0. 0. 0.] [1. 1. 1.]


That was **feature standardisation** in one line. Another example — an outer operation:

In [18]:
a = np.array([1, 2, 3]).reshape(3, 1)   # column
b = np.array([10, 20, 30, 40])          # row
print(a * b)                            # shape (3, 4): every combination

[[ 10  20  30  40]
 [ 20  40  60  80]
 [ 30  60  90 120]]


A shape mismatch gives an error:

In [19]:
try:
    X + np.array([1, 2, 3, 4])
except ValueError as e:
    print("ValueError:", e)

ValueError: operands could not be broadcast together with shapes (4,3) (4,) 


## 7.7 Aggregations and the `axis` argument

In [20]:
M = np.array([[1, 2, 3],
              [4, 5, 6]])
print(M.sum(), M.mean(), M.max(), M.min(), M.std())
print(M.sum(axis=0))    # collapse axis 0 (rows)    -> one value per COLUMN: [5 7 9]
print(M.sum(axis=1))    # collapse axis 1 (columns) -> one value per ROW:    [ 6 15]

21 3.5 6 1 1.707825127659933
[5 7 9]
[ 6 15]


> **Mental model:** `axis=k` is the axis that **disappears**. For `X` of shape `(n_samples, n_features)`, `axis=0` gives per-feature statistics; `axis=1` gives per-sample statistics.

`keepdims=True` keeps the collapsed axis with size 1 — very convenient for broadcasting back:

In [21]:
row_sums = M.sum(axis=1, keepdims=True)    # shape (2, 1)
print(M / row_sums)                        # each row now sums to 1

[[0.16666667 0.33333333 0.5       ]
 [0.26666667 0.33333333 0.4       ]]


`argmax` / `argmin` return the **index** of the extreme value — e.g. turning class probabilities into predicted labels:

In [22]:
probs = np.array([[0.1, 0.7, 0.2],
                  [0.5, 0.3, 0.2],
                  [0.2, 0.2, 0.6]])
print(probs.argmax(axis=1))     # predicted class for each sample
print(np.sort(np.array([3, 1, 2])), np.argsort(np.array([3, 1, 2])))
print(np.unique(np.array([0, 1, 1, 2, 2, 2]), return_counts=True))
print(np.cumsum(np.array([1, 2, 3, 4])))

[1 0 2]
[1 2 3] [1 2 0]
(array([0, 1, 2]), array([1, 2, 3]))
[ 1  3  6 10]


### Missing values: NaN

`np.nan` ("not a number") marks missing data. Any arithmetic with NaN gives NaN, so use the `nan`-aware functions:

In [23]:
v = np.array([1.0, np.nan, 3.0])
print(v.mean(), np.nanmean(v))
print(np.isnan(v))
print(np.nan == np.nan)    # NaN is not equal to anything, even itself!

nan 2.0
[False  True False]
False


## 7.8 Combining arrays

In [24]:
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6]])
print(np.concatenate([a, b], axis=0))      # stack rows
print(np.vstack([a, b]))                   # same
print(np.hstack([a, np.array([[9], [9]])]))  # add a column
print(np.column_stack([np.ones(2), a]))    # e.g. add a bias column of ones
print(np.stack([a, a]).shape)              # NEW axis: (2, 2, 2)

[[1 2]
 [3 4]
 [5 6]]
[[1 2]
 [3 4]
 [5 6]]
[[1 2 9]
 [3 4 9]]
[[1. 1. 2.]
 [1. 3. 4.]]
(2, 2, 2)


## 7.9 Linear algebra

ML is linear algebra. `@` is matrix multiplication (not `*`, which is element-wise!).

In [25]:
A = np.array([[2.0, 1.0],
              [1.0, 3.0]])
v = np.array([1.0, 2.0])

print(A @ v)                 # matrix-vector product
print(A @ A)                 # matrix-matrix product
print(A * A)                 # element-wise: different!
print(v @ v, np.dot(v, v))   # dot product
print(np.linalg.norm(v))     # Euclidean length
print(np.linalg.det(A))
print(np.linalg.inv(A))

[4. 7.]
[[ 5.  5.]
 [ 5. 10.]]
[[4. 1.]
 [1. 9.]]
5.0 5.0
2.23606797749979
5.000000000000001
[[ 0.6 -0.2]
 [-0.2  0.4]]


Solve the linear system $A\mathbf{x} = \mathbf{b}$. Prefer `solve` over computing the inverse — it is faster and numerically more stable:

In [26]:
b = np.array([3.0, 5.0])
x = np.linalg.solve(A, b)
print(x, np.allclose(A @ x, b))

[0.8 1.4] True


Eigenvalues/eigenvectors (the basis of PCA):

In [27]:
eigvals, eigvecs = np.linalg.eig(A)
print(eigvals)
print(eigvecs)

[1.38196601+0.j 3.61803399+0.j]
[[-0.85065081+0.j -0.52573111+0.j]
 [ 0.52573111+0.j -0.85065081+0.j]]


## 7.10 Random numbers

Use the modern **Generator** API with a seed for reproducibility:

In [28]:
rng = np.random.default_rng(seed=42)
print(rng.random(3))                       # uniform [0, 1)
print(rng.integers(0, 10, size=5))         # integers in [0, 10)
print(rng.normal(loc=0, scale=1, size=(2, 3)))
print(rng.choice(["a", "b", "c"], size=4))
print(rng.permutation(5))

[0.77395605 0.43887844 0.85859792]
[0 6 2 0 5]
[[ 0.1278404  -0.31624259 -0.01680116]
 [-0.85304393  0.87939797  0.77779194]]
['c' 'c' 'b' 'b']
[1 0 4 2 3]


(You will also see the older `np.random.seed(0); np.random.rand(...)` style in tutorials; it still works but the Generator is recommended.)

## 7.11 Putting it together: linear regression with NumPy

We generate data from $y = 3 + 2x_1 - x_2 + \text{noise}$ and recover the coefficients with the **normal equations** $\mathbf{w} = (X^\top X)^{-1}X^\top\mathbf{y}$ — solved with `lstsq` for stability — and then with **gradient descent**.

In [29]:
rng = np.random.default_rng(0)
n = 200
X = rng.normal(size=(n, 2))
true_w = np.array([2.0, -1.0])
y = 3 + X @ true_w + rng.normal(scale=0.3, size=n)

X_b = np.column_stack([np.ones(n), X])          # add a column of ones for the intercept
w_closed, *_ = np.linalg.lstsq(X_b, y, rcond=None)
print("closed form:", w_closed.round(3))

# Gradient descent on the MSE loss: grad = (2/n) X^T (Xw - y)
w = np.zeros(3)
lr = 0.1
for step in range(500):
    residual = X_b @ w - y
    grad = 2 / n * X_b.T @ residual
    w -= lr * grad
print("gradient descent:", w.round(3))
print("final MSE:", np.mean((X_b @ w - y) ** 2).round(4))

closed form: [ 2.999  1.985 -0.964]
gradient descent: [ 2.999  1.985 -0.964]
final MSE: 0.0885


Notice: **no Python loop over samples**. Every step processes all 200 samples at once through matrix operations.

---

## Summary

* `ndarray`: fixed type, fast, n-dimensional. Inspect `shape`, `ndim`, `dtype`.
* Index with `a[i, j]`, slice with `a[r0:r1, c0:c1]`, filter with masks `a[a > 0]`.
* Slices are **views**; masks and fancy indexing make **copies**.
* Vectorise: use array operations and ufuncs instead of loops.
* Broadcasting aligns shapes from the right; size-1 dimensions stretch.
* `axis=k` is the axis that disappears; `X.mean(axis=0)` = per-feature mean.
* `@` for matrix multiplication; `np.linalg.solve` for linear systems; `np.random.default_rng(seed)` for randomness.

---

## Exercises

Solutions: `solutions/07_numpy_solutions.ipynb`.

**Exercise 7.1 — Creating arrays.**
Create: (a) a 1-D array of the integers 0–19; (b) reshape it to `(4, 5)`; (c) a `(3, 3)` array of 0.5; (d) 9 evenly spaced numbers from −1 to 1; (e) a `5×5` identity matrix multiplied by 3. Print the shape and dtype of each.

In [30]:
import numpy as np
# Your code here

**Exercise 7.2 — Indexing.**
With `M = np.arange(1, 26).reshape(5, 5)`, extract: (a) the element in the 3rd row, 4th column; (b) the last column; (c) the central `3×3` block; (d) the main diagonal (hint: `np.diag`); (e) all even numbers; (f) the rows whose first element is greater than 10.

In [31]:
M = np.arange(1, 26).reshape(5, 5)
# Your code here

**Exercise 7.3 — Vectorisation.**
Rewrite the loop below as **one line** of NumPy, check both give the same result with `np.allclose`, and time both versions.

In [32]:
rng = np.random.default_rng(1)
x = rng.normal(size=100_000)

result = []
for v in x:
    if v > 0:
        result.append(v ** 2)
    else:
        result.append(0.1 * v)
# Your code here

**Exercise 7.4 — Standardisation and normalisation.**
For the data matrix `X` (rows = samples, columns = features): (a) standardise each column to mean 0 and std 1; (b) min-max scale each column to [0, 1]; (c) normalise each **row** to unit Euclidean length. Verify each result.

In [33]:
rng = np.random.default_rng(2)
X = rng.normal(loc=[10, 50, -3], scale=[2, 15, 0.5], size=(6, 3))
# Your code here

**Exercise 7.5 — Softmax.**
Implement `softmax(z)` for a 2-D array of scores of shape `(n_samples, n_classes)`: $\text{softmax}(z)_j = e^{z_j}/\sum_k e^{z_k}$, computed **per row**. For numerical stability, subtract each row's maximum before exponentiating. Check that every row sums to 1 and that it works for very large scores like `[1000, 1001, 1002]`. Then compute predicted classes.

In [34]:
scores = np.array([[2.0, 1.0, 0.1],
                   [1.0, 3.0, 0.2],
                   [1000.0, 1001.0, 1002.0]])
# Your code here

**Exercise 7.6 — Pairwise distances with broadcasting.**
Given `A` of shape `(m, d)` and `B` of shape `(n, d)`, compute the `(m, n)` matrix of Euclidean distances **without loops** (hint: `A[:, None, :] - B[None, :, :]` has shape `(m, n, d)`). Then, for each row of `A`, find the index of the closest row of `B`.

In [35]:
rng = np.random.default_rng(3)
A = rng.random((4, 2))
B = rng.random((6, 2))
# Your code here

**Exercise 7.7 — Linear regression (challenge).**
Generate `n = 100` points with `x` uniform in [0, 10] and $y = 1.5x^2 - 4x + 2 + \varepsilon$, $\varepsilon \sim N(0, 3^2)$ (seed 0). Fit a **quadratic** model by building the design matrix `[1, x, x^2]` and solving the least-squares problem with `np.linalg.lstsq`. Report the coefficients and the $R^2$.

In [36]:
# Your code here